In [ ]:
# pip install pandas

import pandas as pd

# Büyük veri dosyasını parça parça oku
chunks = pd.read_csv("final_combined_data.csv", chunksize=5000, low_memory=False)

filtered_data = []

for chunk in chunks:
    # Temizleme
    chunk = chunk.replace({'NA': None, 'Unknown': None, 'Not Reported': None})
    chunk['TMB'] = pd.to_numeric(chunk['TMB'], errors='coerce')
    chunk['PD-L1'] = pd.to_numeric(chunk['Clinically reported PD-L1 score'], errors='coerce')

    # Klinik eşiğe göre filtrele (TMB ≥ 3, PD-L1 ≥ 1)
    filtered = chunk[(chunk['TMB'] >= 3) & (chunk['PD-L1'] >= 1)]

    # Gerekli sütunlar eksikse at
    filtered = filtered.dropna(subset=['MSI Type', 'Durable clinical response', 'TMB', 'PD-L1'])

    # Uygun olan satırları listeye ekle
    filtered_data.append(filtered)

# Hepsini birleştir
final_df = pd.concat(filtered_data, ignore_index=True)

# Yeni düzenlenmiş veri setini kaydet
final_df.to_csv("final_filtered_clinical_dataset.csv", index=False)

print("Yeni dosya başarıyla kaydedildi: final_filtered_clinical_dataset.csv")


In [ ]:
# pip install pandas numpy matplotlib scikit-learn shap scikit-fuzzy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report
import shap
import skfuzzy as fuzz
df = pd.read_csv("data/final_filtered_clinical_dataset.csv")

features = ['Clinically reported PD-L1 score', 'TMB', 'MSI Type']
target = 'Durable clinical response'

df = df.replace({'NA': np.nan, 'Unknown': np.nan, 'Not Reported': np.nan})
df['TMB'] = pd.to_numeric(df['TMB'], errors='coerce')
df = df.dropna(subset=features + [target])

le = LabelEncoder()
df['MSI Type'] = le.fit_transform(df['MSI Type'])
df[target] = le.fit_transform(df[target])
X = df[features]
y = df[target]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("\n🔍 Classification Report:\n")
print(classification_report(y_test, y_pred))
explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test[:100], check_additivity=False)

# Genel bar grafiği
if isinstance(shap_values.values, list) or shap_values.values.ndim == 3:
    values = shap_values.values[:, :, 1]
else:
    values = shap_values.values

mean_abs_shap = np.abs(values).mean(axis=0)
feature_names = X.columns.tolist()
top_indices = np.argsort(mean_abs_shap)[-10:][::-1]
top_features = [feature_names[i] for i in top_indices]
top_values = mean_abs_shap[top_indices]

plt.figure(figsize=(8, 5))
plt.barh(top_features[::-1], top_values[::-1])
plt.xlabel("Ortalama |SHAP| Değeri")
plt.title("SHAP Özellik Önem Grafiği (Genel)")
plt.tight_layout()
plt.show()
hasta_indeksi = 1 # ← buradan hastayı seçebilirsin
print(f"\n🎯 Hasta indeksi: {hasta_indeksi}")
shap_values.feature_names = X.columns.tolist()
shap.plots.waterfall(shap_values[hasta_indeksi, :, 1])
pdl1 = df.iloc[hasta_indeksi]['Clinically reported PD-L1 score']
tmb = df.iloc[hasta_indeksi]['TMB']

x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)

pdl1_low = fuzz.trapmf(x_pdl1, [0, 0, 10, 30])
pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_low = fuzz.trapmf(x_tmb, [0, 0, 5, 10])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])

pdl1_l = fuzz.interp_membership(x_pdl1, pdl1_low, pdl1)
pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
tmb_l = fuzz.interp_membership(x_tmb, tmb_low, tmb)
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)

if tmb_h > 0.5:
    if pdl1_h > 0.3:
        suggestion = "🟢 Monoterapi (PD-1 inhibitörü) önerilir."
    else:
        suggestion = "🟡 Kombinasyon (PD-1 + CTLA-4) önerilir."
else:
    suggestion = "🔴 Daha fazla biyobelirteç verisi gerekli."

print("\n📌 Bulanık Mantık Temelli Tedavi Önerisi:")
print(f"PD-L1: {pdl1}, TMB: {tmb}")
print("Öneri:", suggestion)


In [ ]:
# pip install scikit-fuzzy
import pandas as pd
import numpy as np
import skfuzzy as fuzz

# Veri setini yükle
df = pd.read_csv("final_combined_data.csv")
df = df.replace({'NA': None, 'Unknown': None, 'Not Reported': None})
df['TMB'] = pd.to_numeric(df['TMB'], errors='coerce')
df['PD-L1'] = pd.to_numeric(df['Clinically reported PD-L1 score'], errors='coerce')
df = df.dropna(subset=['TMB', 'PD-L1'])

# Bulanık kümeleri tanımla
x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)
pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])

print("\n🟡 Kombinasyon önerisi verilen hastaların indeksleri:")

for i in range(len(df)):
    pdl1 = df.iloc[i]['Clinically reported PD-L1 score']
    tmb = df.iloc[i]['TMB']

    pdl1_l = fuzz.interp_membership(x_pdl1, pdl1_low, pdl1)
    pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
    tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)

    if tmb_h > 0.5 and pdl1_h <= 0.3:
        print(f"→ hasta_indeksi = {i} | PD-L1: {pdl1}, TMB: {tmb}")



In [ ]:
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, accuracy_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_curve)

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# 📄 VERİ YÜKLEME ve TEMİZLEME
df = pd.read_csv("final_filtered_clinical_dataset.csv")
df['Durable clinical response'] = df['Durable clinical response'].replace({
    "Durable clinical benefit": 1,
    "No durable clinical benefit": 0
})
df = df[df['Durable clinical response'].isin([0, 1])].copy()

# 📌 ÖZELLİKLER
features = [
    'Clinically reported PD-L1 score', 'PD-L1', 'TMB', 'MSI Type',
    'Tumor Purity', 'Mutation Count', 'Fraction Genome Altered', 'Cancer Type Detailed'
]
target = 'Durable clinical response'

# 🔧 DÖNÜŞÜMLER
df['TMB'] = pd.to_numeric(df['TMB'], errors='coerce')
df = df.dropna(subset=features + [target])
df['MSI Type'] = LabelEncoder().fit_transform(df['MSI Type'])
df[target] = df[target].astype(int)

X = df[features]
y = df[target]

# 🔨 ÖN İŞLEME
num_features = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

num_pipe = Pipeline([("imputer", SimpleImputer()), ("scaler", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                     ("encoder", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([("num", num_pipe, num_features),
                                  ("cat", cat_pipe, cat_features)])
X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# 🧪 TRAIN/TEST BÖLÜNMESİ
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, stratify=y, random_state=42)

# ⚖️ SADECE EĞİTİM VERİSİNE SMOTE
smote = SMOTE(random_state=42, k_neighbors=1)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# 🔧 scale_pos_weight
neg = sum(y_train_resampled == 0)
pos = sum(y_train_resampled == 1)
scale_pos_weight = neg / pos

# 🌲 MODEL
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                      scale_pos_weight=scale_pos_weight, random_state=42)
model.fit(X_train_resampled, y_train_resampled)
y_proba = model.predict_proba(X_test)[:, 1]
proba_corrected = 1 - y_proba  # class_1 yerine class_0 çıktığı için tersliyoruz
y_pred = model.predict(X_test)

# 🔍 EŞİK = 0.5 DEĞERLENDİRME
print("\n🔍 Classification Report (Threshold = 0.5):\n")
print(classification_report(y_test, y_pred))
print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("📈 ROC AUC Score (terslenmiş):", roc_auc_score(y_test, proba_corrected))

# 🎯 OPTİMUM EŞİK BULMA
prec, recall, thresholds = precision_recall_curve(y_test, proba_corrected)
f1_scores = 2 * (prec * recall) / (prec + recall + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]
print(f"\n📍 Optimum Threshold: {best_thresh:.2f}")
print(f"📈 En yüksek F1-score: {np.max(f1_scores):.3f}")

# 🧪 YENİ EŞİK İLE TAHMİN
y_pred_opt = (proba_corrected > best_thresh).astype(int)

# 🔍 OPTİMUM EŞİK DEĞERLENDİRME
print("\n🔍 Classification Report (Optimum Eşik ile):\n")
print(classification_report(y_test, y_pred_opt))
print("✅ Accuracy (optimum eşik):", accuracy_score(y_test, y_pred_opt))
print("📈 ROC AUC Score (doğru):", roc_auc_score(y_test, proba_corrected))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_opt)
plt.title("Optimum Eşik ile Confusion Matrix")
plt.show()

# 🧠 SHAP
explainer = shap.Explainer(model, X_train_resampled)
X_test_df = pd.DataFrame(X_test[:50], columns=feature_names)
shap_values = explainer(X_test_df, check_additivity=False)
if isinstance(shap_values.values, list) or shap_values.values.ndim == 3:
    shap_values.values = shap_values.values[:, :, 1]

shap.plots.bar(shap_values)
shap.plots.waterfall(shap_values[0])

# 🤖 BULANIK MANTIK TEDAVİ ÖNERİSİ
hasta_indeksi = 6
pdl1 = df.iloc[hasta_indeksi]['Clinically reported PD-L1 score']
tmb = df.iloc[hasta_indeksi]['TMB']

x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)

pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])

pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)

if tmb_h > 0.5:
    if pdl1_h > 0.3:
        suggestion = "🟢 Monoterapi (PD-1 inhibitörü) önerilir."
    else:
        suggestion = "🟡 Kombinasyon (PD-1 + CTLA-4) önerilir."
else:
    suggestion = "🔴 Daha fazla biyobelirteç verisi gerekli."

print("\n📌 Bulanık Mantık Temelli Tedavi Önerisi:")
print(f"PD-L1: {pdl1}, TMB: {tmb}")
print("Öneri:", suggestion)
print(classification_report(y_test, y_pred_opt))



In [ ]:
import numpy as np
import pandas as pd
import skfuzzy as fuzz

# PD-L1 ve TMB bulanık kümeleri
x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)

pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])

# Kombinasyon önerisi verilen hastaların indekslerini topla
kombinasyon_indices = []

for i in df.index:
    try:
        pdl1 = df.loc[i]['Clinically reported PD-L1 score']
        tmb = df.loc[i]['TMB']
        if pd.notnull(pdl1) and pd.notnull(tmb):
            pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
            tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)
            if tmb_h > 0.5 and pdl1_h <= 0.3:
                kombinasyon_indices.append(i)
    except:
        continue

print("🔹 Kombinasyon tedavisi önerilen hasta indeksleri:")
print(kombinasyon_indices)


In [ ]:
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, accuracy_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_curve)

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# 📄 VERİ YÜKLEME ve TEMİZLEME
df = pd.read_csv("final_filtered_clinical_dataset.csv")
df['Durable clinical response'] = df['Durable clinical response'].replace({
    "Durable clinical benefit": 1,
    "No durable clinical benefit": 0
})
df = df[df['Durable clinical response'].isin([0, 1])].copy()
df['TMB'] = pd.to_numeric(df['TMB'], errors='coerce')
df = df.dropna(subset=[
    'Clinically reported PD-L1 score', 'PD-L1', 'TMB', 'MSI Type',
    'Tumor Purity', 'Mutation Count', 'Fraction Genome Altered',
    'Cancer Type Detailed', 'Durable clinical response'
])
df['MSI Type'] = LabelEncoder().fit_transform(df['MSI Type'])
df[target] = df['Durable clinical response'].astype(int)

# 📌 ÖZELLİKLER
features = [
    'Clinically reported PD-L1 score', 'PD-L1', 'TMB', 'MSI Type',
    'Tumor Purity', 'Mutation Count', 'Fraction Genome Altered', 'Cancer Type Detailed'
]
target = 'Durable clinical response'

X = df[features]
y = df[target]

# 🔨 ÖN İŞLEME
num_features = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

num_pipe = Pipeline([("imputer", SimpleImputer()), ("scaler", StandardScaler())])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", num_pipe, num_features),
    ("cat", cat_pipe, cat_features)
])
X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# 🧪 TRAIN/TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, stratify=y, random_state=42)

# ⚖️ SMOTE
smote = SMOTE(random_state=42, k_neighbors=1)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# ⚖️ scale_pos_weight
neg = sum(y_train_resampled == 0)
pos = sum(y_train_resampled == 1)
scale_pos_weight = neg / pos

# 🌲 MODEL
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                      scale_pos_weight=scale_pos_weight, random_state=42)
model.fit(X_train_resampled, y_train_resampled)
y_proba = model.predict_proba(X_test)[:, 1]
proba_corrected = 1 - y_proba  # çünkü class_0 çıkıyor
y_pred = model.predict(X_test)

# 🔍 STANDART EŞİK ANALİZİ
print("\n🔍 Classification Report (Threshold = 0.5):\n")
print(classification_report(y_test, y_pred))
print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("📈 ROC AUC Score (terslenmiş):", roc_auc_score(y_test, proba_corrected))

# 🎯 EŞİK OPTİMİZASYONU
prec, recall, thresholds = precision_recall_curve(y_test, proba_corrected)
f1_scores = 2 * (prec * recall) / (prec + recall + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]
print(f"\n📍 Optimum Threshold: {best_thresh:.2f}")
print(f"📈 En yüksek F1-score: {np.max(f1_scores):.3f}")

# 🔁 Optimum eşikle yeniden sınıflandır
y_pred_opt = (proba_corrected > best_thresh).astype(int)
print("\n🔍 Classification Report (Optimum Eşik ile):\n")
print(classification_report(y_test, y_pred_opt))
print("✅ Accuracy (optimum eşik):", accuracy_score(y_test, y_pred_opt))
print("📈 ROC AUC Score (doğru):", roc_auc_score(y_test, proba_corrected))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_opt)
plt.title("Optimum Eşik ile Confusion Matrix")
plt.show()

# 🧠 SHAP
explainer = shap.Explainer(model, X_train_resampled)
X_test_df = pd.DataFrame(X_test[:50], columns=feature_names)
shap_values = explainer(X_test_df, check_additivity=False)
if isinstance(shap_values.values, list) or shap_values.values.ndim == 3:
    shap_values.values = shap_values.values[:, :, 1]

shap.plots.bar(shap_values)
shap.plots.waterfall(shap_values[0])
shap_values_df = pd.DataFrame(shap_values.values, columns=feature_names)
shap_values_df.mean().sort_values(ascending=False).head(10)


# 🤖 BULANIK MANTIK TEDAVİ ÖNERİSİ
hasta_indeksi = 6  # örnek hasta
pdl1 = df.iloc[hasta_indeksi]['Clinically reported PD-L1 score']
tmb = df.iloc[hasta_indeksi]['TMB']

x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)

pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])

pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)

if tmb_h > 0.5:
    if pdl1_h > 0.3:
        suggestion = "🟢 Monoterapi (PD-1 inhibitörü) önerilir."
    else:
        suggestion = "🟡 Kombinasyon (PD-1 + CTLA-4) önerilir."
else:
    suggestion = "🔴 Daha fazla biyobelirteç verisi gerekli."

print("\n📌 Bulanık Mantık Temelli Tedavi Önerisi:")
print(f"PD-L1: {pdl1}, TMB: {tmb}")
print("Öneri:", suggestion)


In [ ]:
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from scipy.sparse import issparse

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, accuracy_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_curve)

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# 📄 VERİ YÜKLEME ve TEMİZLEME
df = pd.read_csv("final_filtered_clinical_dataset.csv")
df['Durable clinical response'] = df['Durable clinical response'].replace({
    "Durable clinical benefit": 1,
    "No durable clinical benefit": 0
})
df = df[df['Durable clinical response'].isin([0, 1])].copy()
df['TMB'] = pd.to_numeric(df['TMB'], errors='coerce')
df = df.dropna(subset=[
    'Clinically reported PD-L1 score', 'PD-L1', 'TMB', 'MSI Type',
    'Tumor Purity', 'Mutation Count', 'Fraction Genome Altered',
    'Cancer Type Detailed', 'Durable clinical response'
])
df['MSI Type'] = LabelEncoder().fit_transform(df['MSI Type'])
df['Durable clinical response'] = df['Durable clinical response'].astype(int)

# 📌 ÖZELLİKLER
features = [
    'Clinically reported PD-L1 score', 'PD-L1', 'TMB', 'MSI Type',
    'Tumor Purity', 'Mutation Count', 'Fraction Genome Altered', 'Cancer Type Detailed'
]
target = 'Durable clinical response'

X = df[features]
y = df[target]

# 🔨 ÖN İŞLEME
num_features = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

num_pipe = Pipeline([("imputer", SimpleImputer()), ("scaler", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                     ("encoder", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", num_pipe, num_features),
    ("cat", cat_pipe, cat_features)
])
X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# 🧪 TRAIN/TEST
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, stratify=y, random_state=42)

# ⚖️ SMOTE
smote = SMOTE(random_state=42, k_neighbors=1)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# ⚖️ scale_pos_weight
neg = sum(y_train_resampled == 0)
pos = sum(y_train_resampled == 1)
scale_pos_weight = neg / pos

# 🌲 MODEL
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                      scale_pos_weight=scale_pos_weight, random_state=42)
model.fit(X_train_resampled, y_train_resampled)
y_proba = model.predict_proba(X_test)[:, 1]
proba_corrected = 1 - y_proba  # class_1 başarılı
y_pred = model.predict(X_test)

# 📈 STANDART DEĞERLENDİRME
print("\n🔍 Classification Report (Threshold = 0.5):\n")
print(classification_report(y_test, y_pred))
print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("📈 ROC AUC Score (terslenmiş):", roc_auc_score(y_test, proba_corrected))

# 🎯 EŞİK OPTİMİZASYONU
prec, recall, thresholds = precision_recall_curve(y_test, proba_corrected)
f1_scores = 2 * (prec * recall) / (prec + recall + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]
print(f"\n📍 Optimum Threshold: {best_thresh:.2f}")
print(f"📈 En yüksek F1-score: {np.max(f1_scores):.3f}")

y_pred_opt = (proba_corrected > best_thresh).astype(int)
print("\n🔍 Classification Report (Optimum Eşik ile):\n")
print(classification_report(y_test, y_pred_opt))
print("✅ Accuracy (optimum eşik):", accuracy_score(y_test, y_pred_opt))
print("📈 ROC AUC Score (doğru):", roc_auc_score(y_test, proba_corrected))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_opt)
plt.title("Optimum Eşik ile Confusion Matrix")
plt.show()

# 🧠 SHAP
explainer = shap.Explainer(model, X_train_resampled)
X_test_df = pd.DataFrame(X_test[:50], columns=feature_names)
shap_values = explainer(X_test_df, check_additivity=False)
if isinstance(shap_values.values, list) or shap_values.values.ndim == 3:
    shap_values.values = shap_values.values[:, :, 1]
shap.plots.bar(shap_values)
shap.plots.waterfall(shap_values[0])

# ✳️ TEDAVİ BAŞARI OLASILIĞI
hasta_indeksi = 6
orijinal_index = df.index[hasta_indeksi]
hasta_verisi_raw = X.iloc[hasta_indeksi:hasta_indeksi+1]
hasta_transformed = preprocessor.transform(hasta_verisi_raw)
if issparse(hasta_transformed):
    hasta_transformed = hasta_transformed.toarray()

tedavi_basari_olasiligi = model.predict_proba(hasta_transformed)[0][1]
print("\n🧪 Tahmin Edilen Tedavi Başarı Olasılığı:")
print(f"Bu hasta için tedavinin başarılı olma olasılığı: %{tedavi_basari_olasiligi*100:.2f}")

# 📌 GELİŞMİŞ TEDAVİ ÖNERİ SİSTEMİ
hasta = df.iloc[hasta_indeksi]
pdl1 = hasta.get('Clinically reported PD-L1 score', 0)
tmb = hasta.get('TMB', 0)
cancer = str(hasta.get('Cancer Type Detailed', 'unknown')).lower()
msi = str(hasta.get('MSI Type', 'unknown')).lower()
ecog = hasta.get('ECOG', 99)
metastasis = str(hasta.get('Metastatic Site', 'none')).lower()
age = hasta.get('Age', 0)
egfr_mut = hasta.get('EGFR driver', False)
braf_mut = hasta.get('BRAF driver', False)

if msi in ['msi-high', 'msi-h']:
    suggestion = "🟢 Monoterapi (PD-1 inhibitörü) önerilir (MSI-H hasta)."
elif egfr_mut:
    suggestion = "🔴 EGFR mutasyonu pozitif — PD-1 tedavisi sınırlı etkili olabilir."
elif tmb >= 20:
    suggestion = "🟡 Kombinasyon (PD-1 + CTLA-4) önerilir (yüksek TMB)."
elif tmb >= 10 and pdl1 < 1 and any(x in cancer for x in ['lung', 'melanoma', 'colon']) and ecog <= 1:
    suggestion = "🟡 Kombinasyon önerilir (orta TMB, düşük PD-L1, uygun kanser, iyi performans)."
elif pdl1 >= 50 and ecog <= 1:
    suggestion = "🟢 Monoterapi yeterlidir (yüksek PD-L1, iyi ECOG)."
elif age > 75 or ecog >= 2:
    suggestion = "🔴 Kombinasyon tedavi yan etki riski yüksek — monoterapi veya alternatif tedavi düşünülmeli."
elif metastasis != "none":
    suggestion = "🟡 Metastatik hasta — kombine rejim göz önüne alınabilir."
else:
    suggestion = "🔴 Daha fazla biyobelirteç ve klinik değerlendirme gereklidir."

print("\n📌 Klinik Temelli Gelişmiş Tedavi Önerisi:")
print(f"PD-L1: {pdl1}, TMB: {tmb}, MSI: {msi}, ECOG: {ecog}, Metastaz: {metastasis}, Age: {age}")
print("Öneri:", suggestion)


In [ ]:
# 📦 GEREKLİ KÜTÜPHANELER
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import shap
import matplotlib.pyplot as plt
from scipy.sparse import issparse

# 📄 VERİYİ YÜKLE
df = pd.read_csv("merged_final_clinical_dataset.csv")

# 🎯 HEDEF VE ÖZELLİKLER
target = "Durable clinical response"
X = df.drop(columns=[target])
y = df[target]

# 🔧 SÜTUNLARI AYIR
numeric_features = ['TMB', 'Clinically reported PD-L1 score']
categorical_features = ['MSI Normalized', 'Age Group', 'Cancer Group']
binary_features = ['Metastasis Present']

# 📦 ÖN İŞLEME
numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
], remainder='passthrough')  # binary alanlar doğrudan geçer

X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# ⚖️ SMOTE
smote = SMOTE(random_state=42, k_neighbors=1)
X_resampled, y_resampled = smote.fit_resample(X_processed, y)

# 🔀 TRAIN-TEST
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, stratify=y_resampled, random_state=42)

# 🌲 MODEL
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

# 🔍 TAHMİN
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# 📈 METRİKLER
print("\n📊 Classification Report:\n")
print(classification_report(y_test, y_pred))
print("✅ Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("📈 ROC AUC Score:", round(roc_auc_score(y_test, y_proba), 4))
ConfusionMatrixDisplay.from_estimator(model, X_test, y_test)
plt.title("Confusion Matrix")
plt.show()

# 🧠 SHAP ANALİZİ
explainer = shap.Explainer(model, X_train)
X_test_df = pd.DataFrame(X_test[:50].toarray() if issparse(X_test[:50]) else X_test[:50], columns=feature_names)
shap_values = explainer(X_test_df, check_additivity=False)

# 📊 ÖZELLİK ÖNEMİ GRAFİĞİ
shap.plots.bar(shap_values)

# 🔬 TEKİL HASTA ANALİZİ (örnek: 0. hasta)
instance_index = 0
shap.plots.waterfall(shap_values[instance_index])

# 🧪 BAŞARI OLASILIK TAHMİNİ
print("\n📌 Örnek Hasta Başarı Olasılığı:")
sample = X_test_df.iloc[[instance_index]]
sample_proba = model.predict_proba(sample)[0][1]
print(f"Bu hasta için tedavinin başarılı olma olasılığı: %{sample_proba * 100:.2f}")

# 🧠 KLİNİK TEDAVİ ÖNERİSİ
# Orijinal değerleri yakalayalım
raw_patient_index = df.index[instance_index]
raw_patient = df.loc[raw_patient_index]

# Klinik parametreler
pdl1 = raw_patient['Clinically reported PD-L1 score']
tmb = raw_patient['TMB']
age_group = raw_patient['Age Group']
msi = str(raw_patient['MSI Normalized']).lower()
cancer_group = str(raw_patient['Cancer Group']).lower()
metastasis = raw_patient['Metastasis Present']

# Klinik öneri kuralları
if "msi-h" in msi:
    suggestion = "🟢 Monoterapi (PD-1 inhibitörü) önerilir (MSI-H hasta)"
elif tmb >= 20:
    suggestion = "🟡 Kombinasyon (PD-1 + CTLA-4) önerilir (yüksek TMB)"
elif tmb >= 10 and pdl1 < 1:
    suggestion = "🟡 Kombinasyon (PD-1 + CTLA-4) önerilir (orta TMB ve düşük PD-L1)"
elif pdl1 >= 50:
    suggestion = "🟢 Monoterapi önerilir (yüksek PD-L1)"
elif age_group == "Elderly" or metastasis == 1:
    suggestion = "🟡 Kombinasyon tedavi düşünülebilir ancak yaş ve/veya metastaz durumu nedeniyle düşük yan etki riski açısından monoterapi tercih edilebilir"
else:
    suggestion = "🔴 Daha fazla biyobelirteç verisi gerekli"

# Sonucu yazdır
print("\n📌 Klinik Kurallara Dayalı Tedavi Önerisi:")
print(f"PD-L1: {pdl1}, TMB: {tmb}, MSI: {msi}, Age group: {age_group}, Metastaz: {metastasis}")
print("Öneri:", suggestion)


In [ ]:
# 📦 GEREKLİ KÜTÜPHANELER
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, accuracy_score, roc_auc_score,
                             precision_recall_curve, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from scipy.sparse import issparse

# 📄 VERİ YÜKLEME
df = pd.read_csv("synthetic_imbalanced_300_clinical_dataset.csv")
df = df.dropna(subset=['response', 'pd_l1', 'tmb', 'age', 'msi', 'metastasis'])

X = df[['pd_l1', 'tmb', 'age', 'msi', 'metastasis']]
y = df["response"]

# 🔧 ÖZELLİK ÖN İŞLEME
numeric_features = ['tmb', 'pd_l1']
categorical_features = ['age', 'msi']

numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
], remainder='passthrough')

X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# 📤 TRAIN/TEST SPLIT
X_train_raw, X_test_raw, y_train_raw, y_test = train_test_split(
    X_processed, y, test_size=0.2, stratify=y, random_state=42
)

# ⚖️ SMOTE (sadece eğitim verisine)
smote = SMOTE(random_state=42, k_neighbors=1)
X_train, y_train = smote.fit_resample(X_train_raw, y_train_raw)

# 🌲 MODEL EĞİTİMİ
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

# 🔮 TAHMİN
y_proba = model.predict_proba(X_test_raw)[:, 1]

# 🎯 OPTİMUM EŞİK HESABI
prec, recall, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (prec * recall) / (prec + recall + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]

# 🔍 SONUÇLAR
y_pred = (y_proba > best_thresh).astype(int)

print("\n📍 Optimum Eşik Değeri:", round(best_thresh, 2))
print("\n📊 Classification Report (Optimum Eşik):\n")
print(classification_report(y_test, y_pred))
print("✅ Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("📈 ROC AUC Score:", round(roc_auc_score(y_test, y_proba), 4))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("📌 Confusion Matrix (Optimum Threshold)")
plt.show()

# 🧠 SHAP ANALİZİ
explainer = shap.Explainer(model, X_train)
X_test_df = pd.DataFrame(X_test_raw[:50].toarray() if issparse(X_test_raw[:50]) else X_test_raw[:50],
                         columns=feature_names)
shap_values = explainer(X_test_df, check_additivity=False)

shap.plots.bar(shap_values)
shap.plots.waterfall(shap_values[0])

# 🤖 BULANIK MANTIK TEDAVİ ÖNERİSİ
x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)
x_age = np.arange(20, 101, 1)

# Üyelik fonksiyonları
pdl1_high = fuzz.trapmf(x_pdl1, [30, 60, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 15, 50, 50])
age_old = fuzz.trapmf(x_age, [60, 70, 100, 100])

# 🎯 ÖRNEK HASTA ANALİZİ
hasta_indeksi = 72
pdl1 = df.iloc[hasta_indeksi]['pd_l1']
tmb = df.iloc[hasta_indeksi]['tmb']
age_cat = df.iloc[hasta_indeksi]['age']
msi = df.iloc[hasta_indeksi]['msi']
metastasis = df.iloc[hasta_indeksi]['metastasis']

# Yaş dönüştür
age_map = {"Young": 30, "Middle-Aged": 50, "Senior": 70, "Elderly": 80}
age = age_map.get(age_cat, 60)

pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)
age_o = fuzz.interp_membership(x_age, age_old, age)

# 🩺 BULANIK KURAL BAZLI TEDAVİ
if "msi-h" in msi.lower():
    suggestion = "🟢 Monoterapi (MSI-H hasta)"
elif tmb_h > 0.5 and pdl1_h < 0.4:
    suggestion = "🟡 Kombinasyon (TMB yüksek, PD-L1 düşük)"
elif pdl1_h > 0.6 and tmb_h > 0.6:
    suggestion = "🟢 Monoterapi (PD-L1 ve TMB yüksek)"
elif pdl1_h < 0.3 and tmb_h < 0.3:
    suggestion = "🔴 Yetersiz veri – başka biyobelirteçler gerekli"
elif age_o > 0.6 or metastasis == 1:
    suggestion = "🟡 Kombinasyon düşünülebilir, yan etki riski varsa monoterapi tercih edilebilir"
else:
    suggestion = "🔴 Klinik değerlendirme ile karar verilmeli"

# 🧠 MODEL TAHMİNİ
sample_X = X_processed[hasta_indeksi]
proba = model.predict_proba([sample_X])[0][1]

print("\n🧠 Model Tahmini:")
print(f"Bu hasta için tedavi başarılı olma olasılığı: %{proba*100:.2f}")

print("\n📌 Bulanık Mantık Temelli Tedavi Önerisi:")
print(f"PD-L1: {pdl1}, TMB: {tmb}, Yaş: {age_cat}, MSI: {msi}, Metastaz: {metastasis}")
print("Öneri:", suggestion)


In [ ]:
# 📦 GEREKLİ KÜtÜPHANELER
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, accuracy_score, roc_auc_score,
                             precision_recall_curve, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from scipy.sparse import issparse

# 📄 VERİ YÜKLEME
file_path = "synthetic_imbalanced_300_clinical_dataset_enriched.csv"
df = pd.read_csv(file_path)
df = df.dropna()

# 🎯 ÖZELLİKLER
target = "response"
features = ['pd_l1', 'tmb', 'age', 'msi', 'metastasis',
            'ecog_score', 'smoking_status', 'cancer_stage', 'treatment_start_days']

X = df[features]
y = df[target]

# 🔧 ÖN İŞLEME
numeric_features = ['tmb', 'pd_l1', 'ecog_score', 'treatment_start_days']
categorical_features = ['age', 'msi', 'smoking_status', 'cancer_stage']
binary_features = ['metastasis']

numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
], remainder='passthrough')

X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# ⚖️ SMOTE
smote = SMOTE(random_state=42, k_neighbors=1)
X_resampled, y_resampled = smote.fit_resample(X_processed, y)

# 🔀 TRAIN/TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, stratify=y_resampled, random_state=42)

# 🌲 MODEL
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]

# 🌟 THRESHOLD OPTİMİZASYONU
prec, recall, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (prec * recall) / (prec + recall + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]
y_pred_opt = (y_proba >= best_thresh).astype(int)

print(f"\n🔍 Optimum Eşik Değeri: {best_thresh:.2f}")
print("\n📊 Classification Report (Optimum Eşik):\n")
print(classification_report(y_test, y_pred_opt))
print("✅ Accuracy:", round(accuracy_score(y_test, y_pred_opt), 4))
print("📈 ROC AUC Score:", round(roc_auc_score(y_test, y_proba), 4))

# 🔳 CONFUSION MATRIX
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_opt,
                                        display_labels=["Başarısız", "Başarılı"],
                                        cmap="Blues")
plt.title(f"Optimum Eşik ile Confusion Matrix\n(Eşik = {best_thresh:.2f})")
plt.show()

# 🧠 SHAP ANALİZİ
explainer = shap.Explainer(model, X_train)
X_test_df = pd.DataFrame(X_test[:50].toarray() if issparse(X_test[:50]) else X_test[:50],
                         columns=feature_names)
shap_values = explainer(X_test_df, check_additivity=False)
shap.plots.bar(shap_values)
shap.plots.waterfall(shap_values[0])

# 🧐 BULANIK MANTIK
x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)
x_age = np.arange(20, 101, 1)
x_ecog = np.arange(0, 5, 1)
x_delay = np.arange(0, 101, 1)

pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])
age_old = fuzz.trapmf(x_age, [60, 70, 100, 100])
ecog_bad = fuzz.trapmf(x_ecog, [2, 3, 4, 4])
delay_late = fuzz.trapmf(x_delay, [30, 60, 100, 100])

# 🌟 ÖRNEK HASTA
idx = 192
row = df.iloc[idx]

pdl1 = row['pd_l1']
tmb = row['tmb']
age = 80 if row['age'] == "Elderly" else 60
msi = row['msi']
metastasis = row['metastasis']
ecog = row['ecog_score']
delay = row['treatment_start_days']
smoking = row['smoking_status'].lower()
stage = row['cancer_stage']

pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)
age_o = fuzz.interp_membership(x_age, age_old, age)
ecog_b = fuzz.interp_membership(x_ecog, ecog_bad, ecog)
delay_l = fuzz.interp_membership(x_delay, delay_late, delay)

# 🔄 BULANIK KARAR KURALLARI
if "msi-h" in msi.lower():
    suggestion = "🟢 Monoterapi (MSI-H hasta)"
elif tmb_h > 0.6 and pdl1_h < 0.4:
    suggestion = "🟡 Kombinasyon (TMB yüksek, PD-L1 düşük)"
elif pdl1_h > 0.6 and tmb_h > 0.6:
    suggestion = "🟢 Monoterapi (PD-L1 ve TMB yüksek)"
elif pdl1_h < 0.3 and tmb_h < 0.3:
    suggestion = "🔴 Yetersiz veri – başka biyobelirteçler gerekli"
elif age_o > 0.6 or metastasis == 1 or ecog_b > 0.5 or delay_l > 0.5:
    if smoking in ['current smoker', 'former smoker'] or stage in ['III', 'IV']:
        suggestion = "🟡 Kombinasyon (yüksek risk grubu – ileri evre/sigara geçmişi)"
    else:
        suggestion = "🟡 Kombinasyon düşünülebilir, yan etki riski varsa monoterapi"
else:
    suggestion = "🔴 Klinik değerlendirme ile karar verilmeli"

# 🔬 MODEL TAHMİNİ (TEK HASTA)
sample_X = X_processed[idx]
proba = model.predict_proba([sample_X])[0][1]

print("\n Model Tahmini:")
print(f"Bu hasta için tedavi başarılı olma olasılığı: %{proba * 100:.2f}")
print("\n📌 Bulanık Mantık Temelli Tedavi Önerisi:")
print(f"PD-L1: {pdl1}, TMB: {tmb}, ECOG: {ecog}, Gecikme: {delay}, Yaş (grup): {row['age']}, Sigara: {row['smoking_status']}, Evre: {stage}")
print("Öneri:", suggestion)


In [ ]:
# 📦 GEREKLİ KÜTÜPHANELER
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, accuracy_score, roc_auc_score,
                             precision_recall_curve, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# 📄 VERİ YÜKLEME
df = pd.read_csv("synthetic_imbalanced_300_clinical_dataset_enriched.csv").dropna()

# 🎯 ÖZELLİKLER
target = "response"
features = ['pd_l1', 'tmb', 'age', 'msi', 'metastasis',
            'ecog_score', 'smoking_status', 'cancer_stage', 'treatment_start_days']
X = df[features]
y = df[target]

# 🔧 ÖN İŞLEME
numeric_features = ['tmb', 'pd_l1', 'ecog_score', 'treatment_start_days']
categorical_features = ['age', 'msi', 'smoking_status', 'cancer_stage']

numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
], remainder='passthrough')

X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# ⚖️ SMOTE
smote = SMOTE(random_state=42, k_neighbors=1)
X_resampled, y_resampled = smote.fit_resample(X_processed, y)

# 🔀 TRAIN/TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, stratify=y_resampled, random_state=42)

# 🌲 MODEL
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]

# 🌟 THRESHOLD OPTİMİZASYONU
prec, recall, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (prec * recall) / (prec + recall + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]
y_pred_opt = (y_proba >= best_thresh).astype(int)

print(f"\n🔍 Optimum Eşik Değeri: {best_thresh:.2f}")
print("\n📊 Classification Report (Optimum Eşik):\n")
print(classification_report(y_test, y_pred_opt))
print("✅ Accuracy:", round(accuracy_score(y_test, y_pred_opt), 4))
print("📈 ROC AUC Score:", round(roc_auc_score(y_test, y_proba), 4))

# 🔳 CONFUSION MATRIX
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_opt,
                                        display_labels=["Başarısız", "Başarılı"],
                                        cmap="Blues")
plt.title(f"Optimum Eşik ile Confusion Matrix\n(Eşik = {best_thresh:.2f})")
plt.show()

# 👤 İNCELENECEK HASTA
idx = 192  # burayı istediğin hasta ID ile değiştir
row = df.iloc[idx]

# 🔬 MODEL TAHMİNİ
sample_X = X_processed[idx].reshape(1, -1)
proba = model.predict_proba(sample_X)[0][1]


# 🧠 SHAP ANALİZİ (Waterfall)
explainer = shap.Explainer(model, X_train)
hasta_df = pd.DataFrame(sample_X, columns=feature_names)
hasta_shap = explainer(hasta_df, check_additivity=False)
shap.plots.waterfall(hasta_shap[0])  # sadece o hasta için SHAP grafiği

# 🧐 BULANIK MANTIK
x_pdl1 = np.arange(0, 101, 1)
x_tmb = np.arange(0, 51, 1)
x_age = np.arange(20, 101, 1)
x_ecog = np.arange(0, 5, 1)
x_delay = np.arange(0, 101, 1)

pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])
age_old = fuzz.trapmf(x_age, [60, 70, 100, 100])
ecog_bad = fuzz.trapmf(x_ecog, [2, 3, 4, 4])
delay_late = fuzz.trapmf(x_delay, [30, 60, 100, 100])

# 🌟 HASTA VERİSİ
pdl1 = row['pd_l1']
tmb = row['tmb']
age = 80 if row['age'] == "Elderly" else 60
msi = row['msi']
metastasis = row['metastasis']
ecog = row['ecog_score']
delay = row['treatment_start_days']
smoking = row['smoking_status'].lower()
stage = row['cancer_stage']

# 📈 BULANIK ÜYELİK
pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)
age_o = fuzz.interp_membership(x_age, age_old, age)
ecog_b = fuzz.interp_membership(x_ecog, ecog_bad, ecog)
delay_l = fuzz.interp_membership(x_delay, delay_late, delay)

# 🔄 BULANIK KURALLAR
# 🔄 GÜNCELLENMİŞ BULANIK KURALLAR (YUMUŞAK EŞİKLERLE)
# Risk puanı hesapla
risk_score = 0
if age_o > 0.5: risk_score += 1
if ecog_b > 0.5: risk_score += 1
if delay_l > 0.5: risk_score += 1
if metastasis == 1: risk_score += 1
if smoking in ['current smoker', 'former smoker']: risk_score += 1
if stage in ['III', 'IV']: risk_score += 1

# Karar kuralları
if "msi-h" in msi.lower():
    suggestion = "🟢 Monoterapi (MSI-H hasta)"
elif pdl1_h > 0.6 and tmb_h > 0.4 and risk_score <= 2:
    suggestion = "🟢 Monoterapi (PD-L1 ve TMB yüksek, düşük risk)"
elif pdl1_h > 0.6 and tmb_h <= 0.4 and risk_score <= 2:
    suggestion = "🟢 Monoterapi (PD-L1 yüksek, TMB düşük, düşük risk)"
elif tmb_h > 0.6 and pdl1_h < 0.4:
    suggestion = "🟡 Kombinasyon (TMB yüksek, PD-L1 düşük)"
elif pdl1_h < 0.3 and tmb_h < 0.3:
    suggestion = "🔴 Yetersiz veri – başka biyobelirteçler gerekli"
elif risk_score >= 3 and (pdl1_h > 0.4 or tmb_h > 0.4):
    suggestion = "🟡 Kombinasyon (orta-yüksek riskli hasta)"
else:
    suggestion = "🔴 Klinik değerlendirme ile karar verilmeli"

print("\n📌 Bulanık Mantık Temelli Tedavi Önerisi:")
print("\n📊 Model Tahmini:")
print(f"Bu hasta için tedavi başarılı olma olasılığı: %{proba * 100:.2f}")
print(f"PD-L1: {pdl1}, TMB: {tmb}, ECOG: {ecog}, Gecikme: {delay}, Yaş: {row['age']}, Sigara: {row['smoking_status']}, Evre: {stage}")
print("Öneri:", suggestion)


In [ ]:
# 📦 Gerekli kütüphaneler
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_recall_curve
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# 🔧 Fonksiyon: Klinik öneri + fuzzy sistem
def tedavi_onerisi_analizi(row):
    pdl1 = row['pd_l1']
    tmb = row['tmb']
    age = 80 if row['age'] == "Elderly" else 60
    msi = row['msi']
    metastasis = row['metastasis']
    ecog = row['ecog_score']
    delay = row['treatment_start_days']
    smoking = row['smoking_status'].lower()
    stage = row['cancer_stage']

    x_pdl1 = np.arange(0, 101, 1)
    x_tmb = np.arange(0, 51, 1)
    x_age = np.arange(20, 101, 1)
    x_ecog = np.arange(0, 5, 1)
    x_delay = np.arange(0, 101, 1)

    pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
    tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])
    age_old = fuzz.trapmf(x_age, [60, 70, 100, 100])
    ecog_bad = fuzz.trapmf(x_ecog, [2, 3, 4, 4])
    delay_late = fuzz.trapmf(x_delay, [30, 60, 100, 100])

    pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
    tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)
    age_o = fuzz.interp_membership(x_age, age_old, age)
    ecog_b = fuzz.interp_membership(x_ecog, ecog_bad, ecog)
    delay_l = fuzz.interp_membership(x_delay, delay_late, delay)

    risk_score = 0
    if age_o > 0.5: risk_score += 1
    if ecog_b > 0.5: risk_score += 1
    if delay_l > 0.5: risk_score += 1
    if metastasis == 1: risk_score += 1
    if smoking in ['current smoker', 'former smoker']: risk_score += 1
    if stage in ['III', 'IV']: risk_score += 1

    if "msi-h" in msi.lower():
        return "🟢 Monoterapi (MSI-H hasta)"
    elif pdl1_h > 0.6 and tmb_h > 0.4 and risk_score <= 2:
        return "🟢 Monoterapi (PD-L1 ve TMB yüksek, düşük risk)"
    elif pdl1_h > 0.6 and tmb_h <= 0.4 and risk_score <= 2:
        return "🟢 Monoterapi (PD-L1 yüksek, TMB düşük, düşük risk)"
    elif tmb_h > 0.6 and pdl1_h < 0.4:
        return "🟡 Kombinasyon (TMB yüksek, PD-L1 düşük)"
    elif pdl1_h < 0.3 and tmb_h < 0.3:
        return "🔴 Yetersiz veri – başka biyobelirteçler gerekli"
    elif risk_score >= 3 and (pdl1_h > 0.4 or tmb_h > 0.4):
        return "🟡 Kombinasyon (orta-yüksek riskli hasta)"
    else:
        return "🔴 Klinik değerlendirme ile karar verilmeli"

# 📄 Veri yükle
df = pd.read_csv("synthetic_imbalanced_300_clinical_dataset_enriched.csv").dropna()

target = "response"
features = ['pd_l1', 'tmb', 'age', 'msi', 'metastasis',
            'ecog_score', 'smoking_status', 'cancer_stage', 'treatment_start_days']

X = df[features]
y = df[target]

# 🔧 Ön işleme
numeric_features = ['tmb', 'pd_l1', 'ecog_score', 'treatment_start_days']
categorical_features = ['age', 'msi', 'smoking_status', 'cancer_stage']

numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
], remainder='passthrough')

X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# ⚖️ Dengeleme ve modelleme
smote = SMOTE(random_state=42, k_neighbors=1)
X_res, y_res = smote.fit_resample(X_processed, y)

X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, stratify=y_res, random_state=42)
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

# 👤 Hasta ID'si (örnek: 192)
idx = 192
row = df.iloc[idx]
sample_X = X_processed[idx].reshape(1, -1)
proba = model.predict_proba(sample_X)[0][1]

# 🧠 SHAP analiz (senin makinede çalışır)
explainer = shap.Explainer(model, X_train)
hasta_df = pd.DataFrame(sample_X, columns=feature_names)
hasta_shap = explainer(hasta_df, check_additivity=False)
shap.plots.waterfall(hasta_shap[0])  # Hasta için etki analizi

# ✅ Çıktı
print("\n🔬 Klinik Temelli Tedavi Önerisi:")
print(tedavi_onerisi_analizi(row))

print("\n📊 Model Tahmini (XGBoost):")
print(f"Tedavi başarılı olma olasılığı: %{proba * 100:.2f}")
print(f"Hasta ID: {idx} → PD-L1: {row['pd_l1']}, TMB: {row['tmb']}, ECOG: {row['ecog_score']}, Gecikme: {row['treatment_start_days']}")


In [ ]:
# 📦 GEREKLİ KÜTÜPHANELER
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics.pairwise import euclidean_distances
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# 🔄 FONKSİYON: Bulanık Mantık Temelli Tedavi Önerisi
def tedavi_onerisi_analizi(row):
    pdl1 = row['pd_l1']
    tmb = row['tmb']
    age = 80 if row['age'] == "Elderly" else 60
    msi = row['msi']
    metastasis = row['metastasis']
    ecog = row['ecog_score']
    delay = row['treatment_start_days']
    smoking = row['smoking_status'].lower()
    stage = row['cancer_stage']

    x_pdl1 = np.arange(0, 101, 1)
    x_tmb = np.arange(0, 51, 1)
    x_age = np.arange(20, 101, 1)
    x_ecog = np.arange(0, 5, 1)
    x_delay = np.arange(0, 101, 1)

    pdl1_high = fuzz.trapmf(x_pdl1, [30, 50, 100, 100])
    tmb_high = fuzz.trapmf(x_tmb, [5, 10, 50, 50])
    age_old = fuzz.trapmf(x_age, [60, 70, 100, 100])
    ecog_bad = fuzz.trapmf(x_ecog, [2, 3, 4, 4])
    delay_late = fuzz.trapmf(x_delay, [30, 60, 100, 100])

    pdl1_h = fuzz.interp_membership(x_pdl1, pdl1_high, pdl1)
    tmb_h = fuzz.interp_membership(x_tmb, tmb_high, tmb)
    age_o = fuzz.interp_membership(x_age, age_old, age)
    ecog_b = fuzz.interp_membership(x_ecog, ecog_bad, ecog)
    delay_l = fuzz.interp_membership(x_delay, delay_late, delay)

    risk_score = sum([
        age_o > 0.5,
        ecog_b > 0.5,
        delay_l > 0.5,
        metastasis == 1,
        smoking in ['current smoker', 'former smoker'],
        stage in ['III', 'IV']
    ])

    if "msi-h" in msi.lower():
        return "🟢 Monoterapi (MSI-H hasta)"
    elif pdl1_h > 0.6 and tmb_h > 0.4 and risk_score <= 2:
        return "🟢 Monoterapi (PD-L1 ve TMB yüksek, düşük risk)"
    elif pdl1_h > 0.6 and tmb_h <= 0.4 and risk_score <= 2:
        return "🟢 Monoterapi (PD-L1 yüksek, TMB düşük, düşük risk)"
    elif tmb_h > 0.6 and pdl1_h < 0.4:
        return "🟡 Kombinasyon (TMB yüksek, PD-L1 düşük)"
    elif pdl1_h < 0.3 and tmb_h < 0.3:
        return "🔴 Yetersiz veri – başka biyobelirteçler gerekli"
    elif risk_score >= 3 and (pdl1_h > 0.4 or tmb_h > 0.4):
        return "🟡 Kombinasyon (orta-yüksek riskli hasta)"
    else:
        return "🔴 Klinik değerlendirme ile karar verilmeli"

# 📄 VERİ YÜKLE
df = pd.read_csv("synthetic_imbalanced_300_clinical_dataset_enriched.csv").dropna()

features = ['pd_l1', 'tmb', 'age', 'msi', 'metastasis',
            'ecog_score', 'smoking_status', 'cancer_stage', 'treatment_start_days']
target = "response"

X = df[features]
y = df[target]

# 🔧 ÖN İŞLEME
numeric_features = ['tmb', 'pd_l1', 'ecog_score', 'treatment_start_days']
categorical_features = ['age', 'msi', 'smoking_status', 'cancer_stage']

numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# ⚖️ SMOTE
smote = SMOTE(random_state=42, k_neighbors=1)
X_resampled, y_resampled = smote.fit_resample(X_processed, y)

# 🔀 SPLIT ve MODEL
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled)
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
model.fit(X_train, y_train)

# 👤 HASTA ID SEÇ
idx = 123
row = df.iloc[idx]
sample_X = X_processed[idx].reshape(1, -1)

# 🔍 BENZER HASTA KONTROLÜ
distance = euclidean_distances(sample_X, X_processed)[0]
benzerlik_esigi = 1.0
benzer_idx = np.where((distance < benzerlik_esigi) & (distance > 0))[0]
benzer_sayi = len(benzer_idx)
benzer_basari = y.iloc[benzer_idx].mean() if benzer_sayi > 0 else None

# ✅ ÇIKTILAR
print(f"\n📌 Hasta ID: {idx}")
print("\n🧠 Tedavi Önerisi:")
print(tedavi_onerisi_analizi(row))

if benzer_sayi == 0:
    print("\n⚠️ Bu hastaya klinik olarak benzer başka veri bulunamadı.")
    print("🧪 Tahmin düşük güvenilirlikte. Klinik uzman kararı gereklidir.")
else:
    prediction = model.predict_proba(sample_X)[0][1]
    print(f"\n🔍 Benzer hasta sayısı: {benzer_sayi}")
    print(f"📈 Benzer hastalarda başarı ortalaması: %{benzer_basari*100:.2f}")
    print(f"📊 Tedavi Başarı Oranı Tahmini: %{prediction*100:.2f}")

# 📉 SHAP GRAFİĞİ
explainer = shap.Explainer(model, X_train)
hasta_df = pd.DataFrame(sample_X, columns=feature_names)
hasta_shap = explainer(hasta_df, check_additivity=False)
shap.plots.waterfall(hasta_shap[0])